In [0]:
# Store the values from the text input widgets into variables
my_catalog = dbutils.widgets.get("catalog")
my_schema = dbutils.widgets.get("schema")

# Set path to your volume
my_volume_path = f"/Volumes/{my_catalog}/{my_schema}/retail_data"

# display the variables
print(f"my_catalog: {my_catalog}")
print(f"my_schema: {my_schema}")
print(f"my_volume_path: {my_volume_path}")

In [0]:
## --- Data Cleaning ---

## Remove whitespace from column names
df = (
    spark
    .sql(f"""
            SELECT * FROM {my_catalog}.{my_schema}.tb_customer_sales_silver
        """)
    )

df = df.toDF(*[c.replace(' ', '_') for c in df.columns])

## Convert data types for numeric columns and dates
from pyspark.sql.functions import *
from pyspark.sql.types import *

df = (
    df.withColumn("order_date", to_date(df.order_date, "YYYY-MM-DD hh:mm:ss"))
    .withColumn("sale_date", to_date(df.sale_date, "YYYY-MM-DD hh:mm:ss"))
    )

## --- Feature Engineering ---
from pyspark.sql.functions import year, month, date_format, round as spark_round, col, lower, lit

# Extract order year, and order month from order_date column
df = (
    df.withColumn("order_year", year(df.order_date))
    .withColumn("order_month", month(df.order_date))
    .withColumn("order_month_name", date_format(df.order_date, "MMMM"))
    .withColumn("order_day", date_format(df.order_date, "dd"))
)

## 
df.write.mode("overwrite").saveAsTable(f"{my_catalog}.{my_schema}.tb_customer_sales_gold")
 